<a href="https://colab.research.google.com/github/calinvlt/llm-from-scratch/blob/main/Tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import urllib.request
import os
import re

In [2]:
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt")
file_path = "the-verdict.txt"

if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)

In [3]:
# read the file content
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

# print the total number of characters and the first 100 characters
print('Total number of characters:', len(raw_text))
print(raw_text[:99])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
# simple tokenizer
text = 'Hello, world. This is this-- a test?'
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', 'is', 'this', '--', 'a', 'test', '?']


In [5]:
# applying it to the book's text
preprocessed = result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [6]:
# create the vocabulary with special tokens: unknown and end of text
all_words = sorted(set(preprocessed))
all_words.extend(["<|endoftext|>", "<|unk|>"])
vocab_size = len(all_words)
print('Vocabulary size:', vocab_size)

# last 5 words in the vocabulary
print('Last 10 words in the vocabulary:', all_words[-5:])

Vocabulary size: 1132
Last 10 words in the vocabulary: ['younger', 'your', 'yourself', '<|endoftext|>', '<|unk|>']


In [10]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = ' '.join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s([,.:;?_!"()\']|--)\s', r'\1', text)
        return text

In [11]:
# use the SimpleTokenizer class
tokenizer = SimpleTokenizer({s:i for i,s in enumerate(all_words)})
text = """"It's the last he painted you know, "
        Mrs. Gisburn said with a pardonable pride."""
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 115, 754, 793, 7]
" It's the last he painted you know," Mrs.Gisburn said with a pardonable pride .


In [15]:
# Test tokenizer
text1 = "Hello, do you like cake?"
text2 = "In the light of the day."
text = " <|endoftext|> ".join((text1, text2))

print(text)
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

Hello, do you like cake? <|endoftext|> In the light of the day.
[1131, 5, 355, 1126, 628, 1131, 10, 1130, 55, 988, 626, 722, 988, 315, 7]
<|unk|>,do you like <|unk|>?<|endoftext|> In the light of the day .
